# 🤖 Aprofundamento II: IA, Pipelines & Aplicações
**Snowflake Partner Enablement — Lisboa 2026**

---

🎯 **Objetivo:** Demonstrar como o Snowflake transforma dados em insights com IA nativa, pipelines declarativos e aplicações integradas.

⏱️ **Duração:** ~30 minutos

| # | Secção | Tempo | Conceito-chave |
|---|--------|-------|----------------|
| 1 | 🔄 Dynamic Tables | 8 min | ETL declarativo sem orquestração |
| 2 | 🧠 Cortex AI | 12 min | IA nativa em SQL |
| 3 | 🤖 Cortex Agent | 5 min | Orquestração multi-ferramenta |
| 4 | 📱 Apps & Marketplace | 5 min | Partilha e monetização |

---

⚠️ **Pré-requisito:** Este notebook assume que o Aprofundamento I já foi executado (base de dados, tabelas e dados já existem).

---
## 1. 🔄 Dynamic Tables — ETL Declarativo

💡 **Conceito-chave:** Com Dynamic Tables, define-se **O QUÊ** (a transformação) e o Snowflake gere **O QUANDO** e **O COMO** (refresh automático). ETL tão simples como um SELECT.

🔑 **Porquê isto importa:** Os clientes gastam 40-60% do tempo de data engineering em orquestração (Airflow, cron, scripts). Dynamic Tables eliminam essa complexidade por completo.

```
┌────────────────┐     ┌──────────────────────┐     ┌─────────────────┐
│  Tabelas RAW   │ ──► │  Dynamic Tables      │ ──► │  Dashboards/BI  │
│  (fonte)       │     │  (transformação auto) │     │  (consumo)      │
└────────────────┘     └──────────────────────┘     └─────────────────┘
         │                        │
    Dados chegam           Snowflake mantém
    continuamente          tudo atualizado!
```

⚡ `TARGET_LAG = '1 minute'` significa que os dados estarão atualizados no máximo 1 minuto após a fonte mudar.

In [ ]:
%%sql -r dt_customer360
CREATE OR REPLACE DYNAMIC TABLE PARTNERS_LISBON_DEMO.ANALYTICS.customer_360
    TARGET_LAG = '1 minute'
    WAREHOUSE = PARTNERS_WH
AS
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.email,
    c.city,
    c.country,
    c.region,
    c.customer_tier,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COALESCE(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)), 0) AS total_spent,
    COALESCE(AVG(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)), 0) AS avg_order_value,
    MAX(o.order_date) AS last_order_date,
    MODE(p.category) AS favorite_category
FROM PARTNERS_LISBON_DEMO.RAW.customers c
LEFT JOIN PARTNERS_LISBON_DEMO.RAW.orders o ON c.customer_id = o.customer_id
LEFT JOIN PARTNERS_LISBON_DEMO.RAW.order_items oi ON o.order_id = oi.order_id
LEFT JOIN PARTNERS_LISBON_DEMO.RAW.products p ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.email, c.city, c.country, c.region, c.customer_tier;

In [ ]:
%%sql -r dt_sales_category
CREATE OR REPLACE DYNAMIC TABLE PARTNERS_LISBON_DEMO.ANALYTICS.sales_by_category
    TARGET_LAG = '1 minute'
    WAREHOUSE = PARTNERS_WH
AS
SELECT
    p.category,
    DATE_TRUNC('month', o.order_date) AS order_month,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)) AS revenue,
    SUM(oi.quantity) AS units_sold,
    COUNT(DISTINCT o.order_id) AS num_orders
FROM PARTNERS_LISBON_DEMO.RAW.orders o
JOIN PARTNERS_LISBON_DEMO.RAW.order_items oi ON o.order_id = oi.order_id
JOIN PARTNERS_LISBON_DEMO.RAW.products p ON oi.product_id = p.product_id
GROUP BY p.category, DATE_TRUNC('month', o.order_date);

In [ ]:
%%sql -r dt_top_products
CREATE OR REPLACE DYNAMIC TABLE PARTNERS_LISBON_DEMO.ANALYTICS.top_products
    TARGET_LAG = '1 minute'
    WAREHOUSE = PARTNERS_WH
AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.subcategory,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)) AS total_revenue,
    SUM(oi.quantity) AS total_units_sold,
    AVG(r.rating) AS avg_rating,
    COUNT(DISTINCT r.review_id) AS num_reviews
FROM PARTNERS_LISBON_DEMO.RAW.products p
LEFT JOIN PARTNERS_LISBON_DEMO.RAW.order_items oi ON p.product_id = oi.product_id
LEFT JOIN PARTNERS_LISBON_DEMO.RAW.product_reviews r ON p.product_id = r.product_id
GROUP BY p.product_id, p.product_name, p.category, p.subcategory;

### 📊 Consultar as Dynamic Tables

💡 As Dynamic Tables são consultadas exatamente como tabelas normais. O Snowflake garante que os dados estão frescos (dentro do TARGET_LAG definido) — sem precisarmos de acionar nenhum refresh manualmente.

In [ ]:
%%sql -r dt_top_customers
SELECT first_name, last_name, country, customer_tier, total_orders, total_spent
FROM PARTNERS_LISBON_DEMO.ANALYTICS.customer_360
ORDER BY total_spent DESC
LIMIT 10;

In [ ]:
import matplotlib.pyplot as plt

df = dt_top_customers.head(10)

fig, ax = plt.subplots(figsize=(10, 5))
names = df['FIRST_NAME'] + ' ' + df['LAST_NAME']
bars = ax.barh(names, df['TOTAL_SPENT'], color='#29B5E8')
ax.set_title('Top 10 Clientes por Receita Total', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Gasto (€)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x:,.0f}'))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
%%sql -r dt_category_totals
SELECT category, SUM(revenue) AS total_revenue
FROM PARTNERS_LISBON_DEMO.ANALYTICS.sales_by_category
GROUP BY category
ORDER BY total_revenue DESC;

In [ ]:
import matplotlib.pyplot as plt

df = dt_category_totals
colors = ['#29B5E8', '#7D44CF', '#FF9F36', '#71D3DC']

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    df['TOTAL_REVENUE'],
    labels=df['CATEGORY'],
    autopct='%1.1f%%',
    colors=colors[:len(df)],
    textprops={'fontsize': 12}
)
ax.set_title('Distribuição de Receita por Categoria', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Resultado:** 3 Dynamic Tables criadas que:
- Agregam dados de múltiplas tabelas automaticamente
- Mantêm-se atualizadas sem intervenção manual
- São consultadas como tabelas normais

💡 **Zero Airflow, zero cron, zero orquestração manual.** O Snowflake gere tudo.

🔑 Se a fonte muda, a DT atualiza automaticamente dentro do TARGET_LAG. Se **NÃO** muda, **NÃO** consome créditos — eficiência de custos built-in.

---
## 2. 🧠 Cortex AI — IA Nativa em SQL

💡 **Conceito-chave:** O Snowflake traz a IA **AOS DADOS**, não o contrário. Funções de IA chamadas diretamente em SQL — sem APIs externas, sem mover dados, sem infraestrutura ML.

🔑 **Porquê isto importa:** Qualquer analista SQL pode usar IA — não são precisas equipas de ML nem integrações complexas. Os dados **NUNCA saem da plataforma** (ideal para dados sensíveis/regulados).

| Função | O que faz | Exemplo de Uso |
|--------|-----------|----------------|
| SENTIMENT | Score -1 a 1 | Análise de reviews/NPS |
| TRANSLATE | Tradução automática | Localização de conteúdo |
| SUMMARIZE | Resumo de texto | Resumos executivos |
| CLASSIFY_TEXT | Categorização | Routing de tickets |
| COMPLETE | Geração de texto | Marketing, chatbots |

### 2.1 📊 Análise de Sentimento

💡 Numa única query SQL analisamos o sentimento de todas as reviews — sem Python, sem APIs externas, sem infraestrutura adicional.

In [ ]:
%%sql -r cortex_sentiment
SELECT
    review_id,
    product_id,
    rating,
    LEFT(review_text, 60) AS review_preview,
    SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score
FROM PARTNERS_LISBON_DEMO.RAW.product_reviews
ORDER BY sentiment_score;

In [ ]:
import matplotlib.pyplot as plt

df = cortex_sentiment

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#D45B90' if s < 0 else '#FF9F36' if s < 0.5 else '#29B5E8' for s in df['SENTIMENT_SCORE']]
ax.bar(df['REVIEW_ID'].astype(str), df['SENTIMENT_SCORE'], color=colors)
ax.set_title('Score de Sentimento por Review', fontsize=14, fontweight='bold')
ax.set_xlabel('Review ID')
ax.set_ylabel('Sentimento (-1 a 1)')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 2.2 🌍 Tradução (Inglês → Português)

💡 Tradução integrada na plataforma — útil para empresas que recebem feedback em múltiplos idiomas e precisam de normalizar para análise ou reporting local.

In [ ]:
%%sql -r cortex_translate
SELECT
    review_id,
    review_text AS original_en,
    SNOWFLAKE.CORTEX.TRANSLATE(review_text, 'en', 'pt') AS traduzido_pt
FROM PARTNERS_LISBON_DEMO.RAW.product_reviews
LIMIT 5;

### 2.3 📝 Sumarização

💡 Condensar milhares de reviews num único resumo executivo — ideal para dashboards de produto ou relatórios de satisfação.

In [ ]:
%%sql -r cortex_summarize
SELECT SNOWFLAKE.CORTEX.SUMMARIZE(
    LISTAGG(review_text, '. ') WITHIN GROUP (ORDER BY review_id)
) AS summary_of_all_reviews
FROM PARTNERS_LISBON_DEMO.RAW.product_reviews;

### 2.4 🏷️ Classificação de Texto

💡 Classificação automática de texto em categorias predefinidas — imaginem routing automático de tickets de suporte ou categorização de feedback sem regras manuais.

In [ ]:
%%sql -r cortex_classify
SELECT
    review_id,
    LEFT(review_text, 50) AS review_preview,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        review_text,
        ['positive experience', 'negative experience', 'neutral feedback', 'product defect']
    ) AS classification
FROM PARTNERS_LISBON_DEMO.RAW.product_reviews
LIMIT 5;

### 2.5 ✨ LLM Completion (Geração de Texto)

💡 Geração de conteúdo com LLMs (mistral-large2) diretamente via SQL. Os dados de contexto **NUNCA saem do Snowflake** — ideal para empresas com requisitos de privacidade.

In [ ]:
%%sql -r cortex_complete
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'mistral-large2',
    'Based on these product categories: Electronics, Accessories, Audio, Storage - write a brief 2-sentence marketing tagline for a tech accessories store in Lisbon, Portugal.'
) AS generated_text;

✅ **Resultado:** 5 funções de IA, todas em SQL puro:
- 🎭 Sentimento — quantificar feedback
- 🌍 Tradução — localizar conteúdo
- 📝 Sumarização — condensar informação
- 🏷️ Classificação — categorizar automaticamente
- ✨ Geração — criar conteúdo com LLMs

💡 **Tudo isto sem:** APIs externas, chaves OpenAI, infraestrutura ML, movimento de dados.

### 2.6 📊 Semantic View — Perguntas em Linguagem Natural

💡 **Conceito-chave:** A Semantic View cria uma camada semântica que permite ao Cortex Analyst traduzir perguntas em português/inglês para SQL automaticamente.

🔑 **Porquê isto importa:** Democratização de dados — qualquer utilizador de negócio pode obter respostas sem saber SQL.

In [ ]:
%%sql -r semantic_view
CREATE OR REPLACE SEMANTIC VIEW PARTNERS_LISBON_DEMO.ANALYTICS.sv_sales_analytics
    COMMENT = 'Sales analytics semantic view for Partners Lisbon demo'
AS SEMANTIC MODEL
    TABLES (
        cust AS PARTNERS_LISBON_DEMO.RAW.customers PRIMARY KEY (customer_id),
        ord AS PARTNERS_LISBON_DEMO.RAW.orders PRIMARY KEY (order_id),
        items AS PARTNERS_LISBON_DEMO.RAW.order_items PRIMARY KEY (item_id),
        prod AS PARTNERS_LISBON_DEMO.RAW.products PRIMARY KEY (product_id)
    )
    RELATIONSHIPS (
        ord (customer_id) REFERENCES cust (customer_id),
        items (order_id) REFERENCES ord (order_id),
        items (product_id) REFERENCES prod (product_id)
    )
    FACTS (
        ord.order_count AS COUNT(ord.order_id),
        items.total_revenue AS SUM(items.quantity * items.unit_price * (1 - items.discount_pct / 100)),
        items.avg_order_value AS AVG(items.quantity * items.unit_price * (1 - items.discount_pct / 100))
    )
    DIMENSIONS (
        cust.country,
        cust.region,
        cust.customer_tier,
        cust.city,
        prod.category,
        prod.subcategory,
        ord.order_status,
        ord.order_date
    )
    METRICS (
        monthly_revenue AS SUM(items.quantity * items.unit_price * (1 - items.discount_pct / 100))
            GROUP BY DATE_TRUNC('month', ord.order_date)
            COMMENT 'Monthly revenue aggregation',
        customer_ltv AS SUM(items.quantity * items.unit_price * (1 - items.discount_pct / 100))
            GROUP BY cust.customer_id
            COMMENT 'Customer lifetime value'
    );

### 2.7 🔮 ML Functions (Menção Rápida)

💡 **ML sem código** — o Snowflake treina e serve modelos automaticamente, tudo via SQL:

| Função | Caso de Uso |
|--------|-------------|
| `FORECAST` | Previsão de séries temporais |
| `ANOMALY_DETECTION` | Deteção de outliers |
| `CLASSIFICATION` | Classificação supervisionada |
| `TOP_INSIGHTS` | Descoberta automática de padrões |

Estas funções precisam de mais dados para demonstrar eficazmente, mas seguem o mesmo princípio: apenas SQL, zero infraestrutura.

---
## 3. 🤖 Cortex Agent — Orquestração Multi-Ferramenta

💡 **Conceito-chave:** O Cortex Agent combina múltiplas capacidades numa única conversa:
- 📊 **Cortex Analyst** (text-to-SQL via Semantic View)
- 🔍 **Cortex Search** (busca semântica em texto)
- 📈 **Data to Chart** (visualização automática)

🔑 **Porquê isto importa:** Um único agente que responde a perguntas de negócio combinando dados estruturados E não-estruturados. Qualquer utilizador obtém respostas em segundos.

⚡ **Passo 1:** Criar um serviço de busca sobre as reviews
**Passo 2:** Criar o agente que combina tudo

In [ ]:
%%sql -r search_table
CREATE OR REPLACE TABLE PARTNERS_LISBON_DEMO.RAW.reviews_for_search AS
SELECT
    review_id,
    product_id,
    customer_id,
    review_date,
    rating,
    review_text,
    p.product_name,
    p.category
FROM PARTNERS_LISBON_DEMO.RAW.product_reviews r
JOIN PARTNERS_LISBON_DEMO.RAW.products p ON r.product_id = p.product_id;

In [ ]:
%%sql -r cortex_search
CREATE OR REPLACE CORTEX SEARCH SERVICE PARTNERS_LISBON_DEMO.RAW.reviews_search
    ON review_text
    ATTRIBUTES rating, product_name, category
    WAREHOUSE = PARTNERS_WH
    TARGET_LAG = '1 hour'
AS (
    SELECT review_text, rating, product_name, category
    FROM PARTNERS_LISBON_DEMO.RAW.reviews_for_search
);

In [ ]:
%%sql -r cortex_agent
CREATE OR REPLACE CORTEX AGENT PARTNERS_LISBON_DEMO.ANALYTICS.partners_sales_agent
FROM SPECIFICATION $$
models:
  - mistral-large2
orchestration:
  agent_type: cortex_analyst_agent
instructions: |
  You are a sales analytics assistant for a technology products company.
  Answer questions about sales, customers, products, and reviews.
  Use the analyst tool for structured data queries and search for product reviews.
  Always respond in a helpful and concise manner.
tools:
  - tool_type: cortex_analyst_text_to_sql
    tool_spec:
      semantic_view: PARTNERS_LISBON_DEMO.ANALYTICS.sv_sales_analytics
  - tool_type: cortex_search
    tool_spec:
      search_service: PARTNERS_LISBON_DEMO.RAW.reviews_search
  - tool_type: data_to_chart
$$;

### 💬 Testar o Agente

Exemplos de perguntas que podem ser feitas ao agente:
- "Top 5 produtos por receita?"
- "O que dizem os clientes sobre o laptop?"
- "Qual a receita por país?"
- "Mostra-me um gráfico de vendas por categoria"

⚡ Podem testar no Snowsight em **AI & ML > Agents**, ou via SQL abaixo:

In [ ]:
%%sql -r agent_query
SELECT SNOWFLAKE.CORTEX.AGENT(
    'PARTNERS_LISBON_DEMO.ANALYTICS.partners_sales_agent',
    'What are the top 3 product categories by total revenue?'
) AS agent_response;

✅ **Resultado:** Um agente inteligente que:
- Traduz perguntas em SQL (via Semantic View)
- Pesquisa em texto livre (via Cortex Search)
- Gera visualizações (via Data to Chart)

💡 **Sem infraestrutura adicional.** Tudo gerido pelo Snowflake.

---
## 4. 📱 Aplicações & Marketplace

💡 **Conceito-chave:** O Snowflake não é só um data warehouse — é uma plataforma para partilhar e monetizar dados e aplicações. Zero-copy sharing permite partilhar dados sem mover bytes. Native Apps permitem empacotar soluções completas.

🔑 **Porquê isto importa:** O Marketplace é uma oportunidade de **receita recorrente** — empacotar soluções e distribuí-las globalmente sem infraestrutura de entrega.

| Funcionalidade | Benefício |
|----------------|-----------|
| Secure Views | Expor dados sem revelar lógica |
| Zero-Copy Clone | Dev/test sem custo de storage |
| Tags | Classificação automática |
| Native Apps | Solução completa empacotada |
| Marketplace | Distribuição global |

### 🔐 Secure Views para Partilha

💡 Uma Secure View expõe dados agregados sem revelar a lógica de negócio por trás — ideal para partilha com terceiros.

In [ ]:
%%sql -r secure_view
CREATE OR REPLACE SECURE VIEW PARTNERS_LISBON_DEMO.ANALYTICS.partner_analytics_view AS
SELECT
    c.country,
    c.region,
    p.category,
    p.subcategory,
    COUNT(DISTINCT o.order_id) AS num_orders,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)) AS total_revenue,
    AVG(oi.unit_price) AS avg_price
FROM PARTNERS_LISBON_DEMO.RAW.customers c
JOIN PARTNERS_LISBON_DEMO.RAW.orders o ON c.customer_id = o.customer_id
JOIN PARTNERS_LISBON_DEMO.RAW.order_items oi ON o.order_id = oi.order_id
JOIN PARTNERS_LISBON_DEMO.RAW.products p ON oi.product_id = p.product_id
GROUP BY c.country, c.region, p.category, p.subcategory;

### ♻️ Zero-Copy Cloning

💡 Clonagem instantânea — cria ambientes de dev/test em milissegundos. Sem duplicação de dados, sem custo adicional de storage (até haver alterações).

In [ ]:
%%sql -r clone_demo
CREATE OR REPLACE TABLE PARTNERS_LISBON_DEMO.RAW.customers_clone
    CLONE PARTNERS_LISBON_DEMO.RAW.customers;

SELECT COUNT(*) AS clone_count FROM PARTNERS_LISBON_DEMO.RAW.customers_clone;

### 🏷️ Tags e Classificação de Dados

💡 Tags permitem classificar dados para governança automática — por exemplo, aplicar masking automaticamente a TODAS as colunas tagadas como PII.

In [ ]:
%%sql -r tags_demo
CREATE OR REPLACE TAG PARTNERS_LISBON_DEMO.GOVERNANCE.pii_type
    ALLOWED_VALUES 'email', 'phone', 'name', 'address';

CREATE OR REPLACE TAG PARTNERS_LISBON_DEMO.GOVERNANCE.data_sensitivity
    ALLOWED_VALUES 'high', 'medium', 'low';

ALTER TABLE PARTNERS_LISBON_DEMO.RAW.customers
    MODIFY COLUMN email SET TAG PARTNERS_LISBON_DEMO.GOVERNANCE.pii_type = 'email';
ALTER TABLE PARTNERS_LISBON_DEMO.RAW.customers
    MODIFY COLUMN phone SET TAG PARTNERS_LISBON_DEMO.GOVERNANCE.pii_type = 'phone';
ALTER TABLE PARTNERS_LISBON_DEMO.RAW.customers
    MODIFY COLUMN email SET TAG PARTNERS_LISBON_DEMO.GOVERNANCE.data_sensitivity = 'high';

In [ ]:
%%sql -r show_tags
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'PARTNERS_LISBON_DEMO.RAW.CUSTOMERS', 'TABLE'
));

✅ **Resultado:**
- 🔐 Secure View — dados partilhados sem expor lógica
- ♻️ Clone — ambiente de dev em milissegundos
- 🏷️ Tags — classificação para governança automática

💡 **Oportunidade:**
- Criar **Native Apps** com soluções completas
- Publicar no **Snowflake Marketplace** (2000+ datasets e apps)
- Receita recorrente por consumo

---
## 🎯 Resumo — Aprofundamento II

| Capacidade | Valor |
|------------|---------------------|
| 🔄 Dynamic Tables | Pipelines sem orquestração |
| 🧠 Cortex AI | IA em SQL, sem infraestrutura |
| 🤖 Cortex Agent | Assistentes inteligentes prontos a usar |
| 📱 Apps & Marketplace | Escalar soluções globalmente |

---

🔑 **Mensagens-chave:**
1. **IA nativa** = dados nunca saem da plataforma
2. **Pipelines declarativos** = menos código, mais valor
3. **Agentes inteligentes** = democratização total de dados
4. **Marketplace** = oportunidade de negócio para partners

**👉 Próximo:** Laboratório Prático — Mãos na massa!